# Evaluación 1

**Integrante 1:** Cristina Morán

**Integrante 2:** Alonso Valderrama

**Integrante 3:** Amasis Guzmán

**Correo Electrónico integrante 1:** cristina.moran2201@alumnos.ubiobio.cl

**Correo Electrónico integrante 2:** alonso.valderrama2201@alumnos.ubiobio.cl

**Correo Electrónico integrante 3:** amasis.guzman2201@alumnos.ubiobio.cl

**Fecha de Creación:** Abril de 2026  
**Versión:** 1.0  

---

## Descripción

Este notebook contiene el desarrollo de la evaluación 1 de la asignatura Inteligencia Artificial de la carrera de Ingeniería Civil en Informática de la Universidad del Bio Bio, sede Concepción.


---

## Requisitos de Software

Este notebook fue desarrollado con Python 3.12. A continuación se listan las bibliotecas necesarias:

- pandas (>=1.1.0)
- matplotlib (3.7.1)
- numpy (2.0.2)


Para verificar la versión instalada ejecutar usando el siguiente comando, usando la librería de la cual quieres saber la versión:

```python
import pandas as pd
print(pd.__version__)
````

#Carga de datos

In [ ]:
!wget https://raw.githubusercontent.com/JaznaLaProfe/datos/master/preparation_data/dataset_churn_dirty.csv

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sb

from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder

In [ ]:
#Carga el set de datos
data = pd.read_csv('dataset_churn_dirty.csv')

#Para mostrar en una tabla los datos del archivo
data.head() 

In [ ]:
# Cantidad de observaciones y columnas
data.shape

Comentario: Esto quiere decir que los datos del archivo \.csv ingresado poseen 16 columnas y cada una de ellas 21000 valores\.

# Revisión de nulos

In [ ]:
# revisión de valores faltantes y tipos de datos
data.info()


De estos datos podemos observar la siguente cantidad de datos nulos:

age : 1036 datos nulos\.

monthly\_income: 1038 datos nulos\.

payment\_method : 1047 datos nulos\.

Estos datos se mostraran en la siguiente tabla con su % de nulos :

gender: 1053 datos nulos\.

Comentario respecto a nulos: Se puede apreciar que las 4 columnas con nulos son "age", "monthly\_income", "gender" y "payment\_method", además que la cantidad de datos total de cada columna debiera ser de 21000 entradas\.

In [ ]:
columnas_con_nulos = data.isna().sum()[data.isna().sum() > 0]
porcentaje_nulos = (columnas_con_nulos / data.shape[0]) * 100

resultado = pd.DataFrame({
    "Cantidad Nulos": columnas_con_nulos,
    "Porcentaje Nulos (%)": porcentaje_nulos
}).round(2)

resultado

Observamos que entre estas 4 columnas tan solo el 5% o menos son nulos\.

# Revisión de atípicos

In [ ]:
# Funciones para revisión de atípicos
def buscar_atipicos(data : pd.DataFrame, columna : str) -> pd.DataFrame:
  """
  Busca valores atípicos en una columna.
  """
  # Calcular los límites
  Q1 = data[columna].quantile(0.25)
  Q3 = data[columna].quantile(0.75)
  # Calcula rango intercuartilico
  IQR = Q3 - Q1
  limite_inferior = Q1 - 1.5 * IQR 
  limite_superior = Q3 + 1.5 * IQR

  # Filtrar outliers
  return data[(data[columna] < limite_inferior) | (data[columna] > limite_superior)]

def obtener_cantidad_atipicos(data : pd.DataFrame, columnas : np.array) -> dict:
  """
  Obtiene la cantidad de atípicos por cada columna.
  """
  total_atipicos = {}
  for columna in data[columnas]:
    atipicos = buscar_atipicos(data, columna)
    total_atipicos[columna] = atipicos.shape[0]
  return total_atipicos


In [ ]:
atipicos_por_columna = obtener_cantidad_atipicos(data, data.describe().columns)
atipicos_por_columna

# Existencia de valores atípicos \(outliers\)

Analizando los datos podemos identificar la presencia de valores atípicos en múltiples variables, siendo las más relevantes: age, monthly\_income, num\_logins\_last\_month, avg\_session\_time, support\_tickets, account\_balance y last\_payment\_amount\.
 

In [ ]:
# Columnas numéricas para revisar atípicos
revision_atipicos = [
    'age',
    'monthly_income',
    'tenure_months',
    'num_logins_last_month',
    'avg_session_time',
    'support_tickets',
    'account_balance',
    'last_payment_amount'
]

# Crear subplots (2 filas x 4 columnas porque son 8 variables)
fig, axes = plt.subplots(2, 4, figsize=(20,10))
axes = axes.flatten()

# Graficar boxplots
for i, col in enumerate(revision_atipicos):
    data[col].plot(kind='box', ax=axes[i])
    axes[i].set_title(col)
    axes[i].tick_params(axis="x", labelrotation=45)

# Título general
plt.suptitle("Análisis de valores atípicos", fontsize=16, fontweight="bold")

# Ajuste de diseño
plt.tight_layout()

plt.show()

age \(edad\)
\- Mayoría de las edades entre los 30 y 60 años
\- Mediana está cerca de los 40\-50
\- Existe un error o dato poco realista, valor atípico muy alto \(~140 años\)
\- Existen valores bajos, cercanos a 0
Conclusión: Hay outliers evidentes, especialmente un extremo

monthly\_income \(ingreso mensual\)
\- La mayoría de los ingresos están concentrados en un rango bajo\-medio
\- Existen valores atípicos, incluyendo uno muy alto \(~10 millones\)
\- Existen valores negativos o muy bajos
Conclusión: es una variable con alta dispersión y outliers extremos, probablemente necesita limpieza o transformación\.

EXPLICACIÓN DE CADA VARIABLE:

tenure\_months \(antigüedad en meses\)
\- Los valores están entre 0 y ~120 meses
\- La mediana está cerca de los 60 meses
\- No se observan muchos outliers
Conclusión: Distribución mayoritariamente normal y estable

num\_logins\_last\_month \(número de inicios de sesión\)
\- La mayoría de los usuarios tienen entre 25 y 35 logins
\- Existen outliers tanto Altos \(~50\-55 logins\) y Bajos \(~10\-15 logins\)
Conclusión: Existe variabilidad en el comportamiento de usuarios

avg\_session\_time \(tiempo promedio de sesión\)
\- La mayoría está entre 50 y 80
\- Existen Valores negativos \(posible error\) y un outlier extremo \(~500\)
Conclusión: Existencia de datos inconsistentes \(probablemente errores o registros mal capturados\)

account\_balance \(saldo de cuenta\)
\- La mayoría está entre 30,000 y 60,000\.
Existen valores negativos \(cuentas en deuda\)y valores muy altos \(~100,000\+\)
Conclusión: Presencia clara de outliers en ambos extremos\.

support\_tickets \(tickets de soporte\)
\- La mayoría de los usuarios tiene entre 1 y 3 tickets
\- Algunos valores llegan hasta 7–9 \(outliers\)
Conclusión: Distribución razonable, pero con algunos clientes que requieren mucho soporte

In [ ]:
data.drop(columns=['customer_id']).describe()
# Muestra las medidas estadísticas, descartando la columna que identifica al cliente.

In [ ]:
data.describe(include="object")

# Revisión de inconsistencias

In [ ]:
# Detectar inconsistencias
inconsistentes = data[
    (data["age"] <= 0) |
    (data["monthly_income"] < 0) |
    (data["tenure_months"] < 0) |
    (data["num_logins_last_month"] < 0) |
    (data["avg_session_time"] < 0) |
    (data["support_tickets"] < 0) |
    (data["account_balance"] < 0) |
    (data["last_payment_amount"] < 0)
]

print("Registros inconsistentes encontrados:")
inconsistentes

Comentario: Tabla de inconsistencias totales\. Aquí podemos ver claramente la existencia de datos inconsistentes como en age que tenemos un \-5, monthly\_income = \-100000, avg\_session\_time con un valor negativo\.

# Revisión de inconsistencias para cualitativas

Género

In [ ]:
data.gender.unique()

In [ ]:
data.subscription_type.unique()

Región

In [ ]:
data.region.unique()

Método de Pago

Tipo de Subscripción

In [ ]:
data.payment_method.unique()

Es activo

Dispositivo favorito

In [ ]:
data.preferred_device.unique()

# Revisión de duplicados

In [ ]:
# Revisa la existencia de duplicados
data.duplicated().sum()

Esto quiere decir que existen 1000 datos duplicados, por lo tanto cuando se realice la limpieza en lugar de 21000 datos solo contaremos con 20000 o menos

In [ ]:
# Obtiene los registos duplicados
data[data.duplicated(keep=False)].sort_values(by='customer_id')

Gracias a la variable customer\_id podemos identificar los datos duplicados, los cuales confirman la identidad entre datos\.

# Limpieza y transformación

Comentario respecto a la limpieza de duplicados: Con esta función podemos eliminar los duplicados \(originalmente 1000 datos duplicados\) de los 21000 datos originales, luego de la limpieza la cantidad de datos se redujo\.

Limpieza de Duplicados:

Limpieza Nulos:

Primero, volvemos a visualizar los nulos para comparar

In [ ]:
data = data.drop_duplicates()
data.shape
# Con esto eliminamos los duplicados (1000 datos) del total (21000), dandonos un total de 20000 datos.

In [ ]:
columnas_con_nulos = data.isna().sum()[data.isna().sum() > 0]
porcentaje_nulos = (columnas_con_nulos / data.shape[0]) * 100

resultado = pd.DataFrame({
    "Cantidad Nulos": columnas_con_nulos,
    "Porcentaje Nulos (%)": porcentaje_nulos
}).round(2)

resultado

Volvemos a revisar los nulos, después de la limpieza de duplicados para observar como se vio alterada\. Podemos observar que disminuyeron la cantidad de datos Nulos\.

age                                        1036                                     984

monthly\_income                   1038                                     985

gender                                  1053                                     1000

payment\_method                 1047                                     1000

Diferenciamos las variables cualitativas de las cuantitativas agrupándolas en dos arreglos diferentes para manejarlas de forma correcta más adelante

In [ ]:
# Define las variables cualitativas y cuantitativas disponibles
numeric_features = ['age', 'monthly_income', 'tenure_months', 'num_logins_last_month', 'avg_session_time', 'support_tickets', 'account_balance', 'last_payment_amount']
categorical_features = ['gender', 'payment_method', 'preferred_device', 'region', 'subscription_type', 'is_active']

                                   Antes de la Limpieza    Después de la Limpieza

Creamos la clase Windsorizer para definir su función

Luego de definir Windsorizer podemos definir los Pipeline tanto para variables cuantitativas como cualitativas

## Pipeline Variables Cuantitativas:

In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin): #le entregas los minimos o utiliza los default
  """
  Tratamiento de atípicos

  Parámetros
  ----------
  BaseEstimator : Clase base para estimadores en scikit-learn.
  TransformerMixin : Clase base para transformadores en scikit-learn.

  Atributos
  ---------
  columns_ : array-like
    Nombres de las columnas a transformar.
  limits : tuple
    % de los extremos a descartar
  """
  def __init__(self, limits=(0.05, 0.05)):
    self.limits = limits

  def fit(self, X, y=None):
    # Guardar nombres si es DataFrame, si no generar nombres genéricos
    if isinstance(X, pd.DataFrame):
      self.columns_ = X.columns
    else:
      self.columns_ = np.arange(X.shape[1])
    return self

  def transform(self, X):
    X = pd.DataFrame(X, columns=self.columns_)
    for col in self.columns_:
      lower = X[col].quantile(self.limits[0])
      upper = X[col].quantile(1 - self.limits[1])
      X = X.astype("float64")
      X[col] = np.clip(X[col], lower, upper)
    return X

  def get_feature_names_out(self, input_features=None):
    if input_features is None:
      return np.array(self.columns_)
    else:
      return np.array(input_features)

In [ ]:
# Define el pipeline para las variables cuantitativas
pipeline_numerico = Pipeline(
    steps=[
        ("winsorizer", Winsorizer()),
        ("imputacion", SimpleImputer(strategy="mean")),
        ("escalado", StandardScaler())
    ]
)

## Pipeline Variables Cualitativas:

Variables numéricas \(age, monthly\_income\) → se imputa con la media

In [ ]:
# Define el pipeline para las variables cualitativas
pipeline_categorico = Pipeline(steps=[
    ("imputacion", SimpleImputer(strategy="most_frequent")),
    ("codificacion", OneHotEncoder(handle_unknown="ignore"))
])

Integramos ambos pipeline

In [ ]:
# Integra ambos pipelines
preprocesador = ColumnTransformer(
    transformers=[
        ("num", pipeline_numerico, numeric_features),
        ("cat", pipeline_categorico, categorical_features)
    ]
)

In [ ]:
data.info()

In [ ]:
# Estandariza los strings mal escritos (corrige mayúsculas y minúsculas)
data["gender"] = data["gender"].str.strip().str.capitalize()
data["payment_method"] = data["payment_method"].str.strip().str.lower()
data["preferred_device"] = data["preferred_device"].str.strip().str.lower()
data["region"] = data["region"].str.strip().str.lower()
data["subscription_type"] = data["subscription_type"].str.strip().str.lower()
# Falta esta línea:
data["is_active"] = data["is_active"].str.strip().str.lower()

In [ ]:
# Aplicar el preprocesador
X = data.drop(columns=["churn", "customer_id"])
y = data["churn"]
data_transformada = preprocesador.fit_transform(X)

In [ ]:
# Limpia las inconsistencias encontradas
data = data[
    (data["age"] > 0) &
    (data["age"] < 120) &
    (data["monthly_income"] >= 0) &
    (data["avg_session_time"] >= 0) &
    (data["account_balance"] >= 0) &
    (data["last_payment_amount"] >= 0) &
    (data["tenure_months"] >= 0) &
    (data["num_logins_last_month"] >= 0) &
    (data["support_tickets"] >= 0)
]

# Mostrar la nueva forma del DataFrame para verificar el resultado
data.shape

In [ ]:
data.is_active.unique()

# Guarda set de datos limpio y transformado

In [ ]:
# Opción A: guardar el DataFrame limpio (antes del pipeline)
# data.to_csv("dataset_churn_clean.csv", index=False)

# Opción B: guardar la matriz transformada como DataFrame
columnas_out = preprocesador.get_feature_names_out()
data_final = pd.DataFrame(data_transformada, columns=columnas_out)
data_final.to_csv("dataset_churn_transformado.csv", index=False)

In [ ]:
data.info()

Observamos que la mayoría redujo su cantidad a 16883, pero para las columnas "gender" y "payment\_method" aún no se cuadran los datos

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=412dc4ce-25a6-448b-a3d9-1501e633b989' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>